# 🚁 AgTech Drone — Precision Irrigation Sensor Fusion

**Combines RGB + Thermal drone imagery to identify water-stressed crops.**

This notebook is **self-contained** — just run all cells in order.

| Cell | What it does |
|------|--------------|
| 1    | Install dependencies |
| 2    | Clone the repository |
| 3    | Run a quick headless smoke test |
| 4    | Launch the interactive Gradio dashboard |

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install opencv-python-headless gradio PyYAML --quiet
print('✅  Dependencies installed')

In [ ]:
# ── Cell 2: Clone the repository ─────────────────────────────────────────
import os

REPO = 'agtech-drone-irrigation'
if not os.path.exists(REPO):
    !git clone https://github.com/your-org/agtech-drone-irrigation.git

os.chdir(REPO)
print(f'✅  Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 3: Smoke test (headless) ────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from src.data.synthetic import generate_synthetic_arrays
from src.pipeline.fusion import process_precision_irrigation

rgb, thermal = generate_synthetic_arrays(image_size=500)
overlay, heatmap, n_stress, n_ok = process_precision_irrigation(
    rgb, thermal, temp_threshold=30.0
)

print(f'✅  Pipeline ran successfully')
print(f'   Stressed plants : {n_stress}')
print(f'   Healthy plants  : {n_ok}')
print(f'   Overlay shape   : {overlay.shape}')
assert n_stress == 5 and n_ok == 5, f'Unexpected counts: {n_stress}/{n_ok}'

In [ ]:
# ── Cell 4: Quick matplotlib preview (no Gradio needed) ──────────────────
import cv2
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ['RGB Input', 'Canopy Heatmap', 'Diagnostic Overlay']
imgs   = [rgb, heatmap, overlay]

for ax, title, img in zip(axes, titles, imgs):
    display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
    ax.imshow(display)
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.savefig('preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅  Preview saved as preview.png')

In [ ]:
# ── Cell 5: Launch the Gradio dashboard ──────────────────────────────────
#
# This will print a public share URL (valid for 72 hours).
# Click it to open the interactive dashboard in a new tab.
#
# Controls:
#   • Click '🎲 Load Demo' to auto-populate both image slots
#   • Adjust the threshold slider and click '🔍 Run Analysis'
#   • Upload your own RGB + Thermal images for real-data testing

from src.dashboard.app import build_dashboard

app = build_dashboard()
app.launch(
    share=True,    # Generates a public gradio.live URL for Colab
    inline=True,   # Also embeds the UI in the cell output
    debug=False,
)